In [0]:

%pip install azure-eventhub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json
import random
import time
import uuid
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

CONNECTION_STR = "<YOUR_EVENT_HUB_CONNECTION_STRING>"
EVENTHUB_NAME = "clickstream-events"

CATEGORIES = ["electronics", "fashion", "home", "sports", "books", "beauty", "toys"]
EVENT_TYPES = ["page_view", "add_to_cart", "checkout_start", "purchase"]
EVENT_WEIGHTS = [0.55, 0.25, 0.12, 0.08]

NUM_USERS = 500
NUM_PRODUCTS = 200

def generate_event():
    event_type = random.choices(EVENT_TYPES, weights=EVENT_WEIGHTS, k=1)[0]
    category = random.choice(CATEGORIES)
    price = round(random.uniform(5, 500), 2)

    return {
        "event_id": str(uuid.uuid4()),
        "user_id": f"user_{random.randint(1, NUM_USERS)}",
        "session_id": f"session_{random.randint(1, NUM_USERS * 3)}",
        "event_type": event_type,
        "product_id": f"product_{random.randint(1, NUM_PRODUCTS)}",
        "category": category,
        "price": price if event_type in ["add_to_cart", "checkout_start", "purchase"] else None,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

producer = EventHubProducerClient.from_connection_string(
    conn_str=CONNECTION_STR,
    eventhub_name=EVENTHUB_NAME
)

print("Starting event generator... Click the Stop button on this cell to stop.")

try:
    while True:
        event_batch = producer.create_batch()
        event = generate_event()
        event_batch.add(EventData(json.dumps(event)))
        producer.send_batch(event_batch)
        print(f"Sent: {event['event_type']} | {event['user_id']} | {event['category']}")
        time.sleep(random.uniform(0.5, 2))
except KeyboardInterrupt:
    print("Stopped.")
finally:
    producer.close()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
print("bron:", spark.sql("SELECT COUNT(*) FROM clickstream_catalog.bronze.raw_events").collect()[0][0])
print("silv:", spark.sql("SELECT COUNT(*) FROM clickstream_catalog.silver.clean_events").collect()[0][0])
print("Gold2:", spark.sql("SELECT COUNT(*) FROM clickstream_catalog.gold.category_metrics_by_minute").collect()[0][0])

bron: 4710
silv: 4710
Gold2: 1906
